# 🔍 完全ローカル RAG サーバー
## llama.cpp + OpenSearch + Cloudflare Tunnel
### API キー不要 / Docker 不要 / GPU 不要 / アカウント不要

```
[ブラウザ / curl]
      ↓
[RAG サーバー FastAPI :8100]  ← Cloudflare Tunnel で外部公開
      ↓               ↓
[OpenSearch :9200]  [llama.cpp :18080]
```

| サービス | ポート | 役割 |
|---|---|---|
| llama.cpp LLM | 18080 | テキスト生成（OpenAI 互換） |
| llama.cpp Embed | 18081 | 埋め込みベクトル生成 |
| OpenSearch | 9200 | ドキュメント検索 |
| **RAG サーバー** | **8100** | **ユーザーの窓口（FastAPI）** |

### ⚠️ 注意
- RAM 12GB 以上推奨
- CPU のみの場合、LLM 応答は 10〜30 秒かかります
- ngrok の無料アカウント（Authtoken）が必要です

---
## ① 環境確認

In [ ]:
import sys, subprocess, multiprocessing, shutil

print(f"Python    : {sys.version.split()[0]}")
print(f"CPU コア数 : {multiprocessing.cpu_count()}")

mem = subprocess.run(['free', '-h'], capture_output=True, text=True).stdout
for line in mem.strip().split('\n')[:2]:
    print(line)

disk = subprocess.run(['df', '-h', '/'], capture_output=True, text=True).stdout
for line in disk.strip().split('\n')[:2]:
    print(line)

if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                         capture_output=True, text=True)
    if gpu.returncode == 0:
        print('GPU:', gpu.stdout.strip())
else:
    print('GPU: なし（CPU モードで動作します）')

---
## ② llama.cpp のインストール

In [ ]:
%%bash
pip install -q 'llama-cpp-python[server]' \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
python -c "import llama_cpp; print('✅ llama-cpp-python', llama_cpp.__version__)"
pip install -q huggingface_hub fastapi uvicorn
echo "✅ 全パッケージインストール完了" 

---
## ③ モデルのダウンロード

| モデル | サイズ | 用途 |
|---|---|---|
| Qwen2.5-3B-Instruct Q4_K_M | ~2GB | LLM（ツールコール対応） |
| nomic-embed-text-v1.5 Q4_K_M | ~80MB | 埋め込み |

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.makedirs('/tmp/models', exist_ok=True)

print('LLM モデルをダウンロード中... （約 2GB）')
LLM_PATH = hf_hub_download(
    repo_id='Qwen/Qwen2.5-3B-Instruct-GGUF',
    filename='qwen2.5-3b-instruct-q4_k_m.gguf',
    local_dir='/tmp/models',
)
print('✅ LLM:', LLM_PATH)

print('\n埋め込みモデルをダウンロード中... （約 80MB）')
EMBED_PATH = hf_hub_download(
    repo_id='nomic-ai/nomic-embed-text-v1.5-GGUF',
    filename='nomic-embed-text-v1.5.Q4_K_M.gguf',
    local_dir='/tmp/models',
)
print('✅ Embed:', EMBED_PATH)

---
## ④ llama.cpp LLM サーバーの起動（port 18080）

> Colab 内部の node プロセスが 8080 を占有しているため 18080 を使用します

In [ ]:
import subprocess, sys, os, time, urllib.request, glob

model_files = glob.glob('/tmp/models/*.gguf')
LLM_PATH  = next((f for f in model_files if 'embed' not in f.lower()), None)
LLM_PORT  = 18080

subprocess.run(['fuser', '-k', str(LLM_PORT) + '/tcp'], capture_output=True)
time.sleep(1)

r = subprocess.run([sys.executable, '-m', 'llama_cpp.server', '--help'],
                   capture_output=True, text=True)
help_text = r.stdout + r.stderr
use_n_gpu = '--n_gpu_layers' in help_text or '--n-gpu-layers' in help_text
use_fmt   = '--chat_format'  in help_text or '--chat-format'  in help_text

cmd = [sys.executable, '-m', 'llama_cpp.server',
       '--model', LLM_PATH, '--host', '0.0.0.0', '--port', str(LLM_PORT),
       '--n_ctx', '2048', '--n_threads', '2']
if use_n_gpu: cmd += ['--n_gpu_layers', '0']
if use_fmt:   cmd += ['--chat_format', 'chatml']

llm_proc = subprocess.Popen(cmd,
    stdout=open('/tmp/llm.log', 'w'), stderr=subprocess.STDOUT)

print('LLM サーバー起動中... (最大 60 秒)')
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:' + str(LLM_PORT) + '/v1/models', timeout=3)
        print('\n✅ LLM サーバー起動完了！ port=' + str(LLM_PORT))
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(2)
else:
    print('\n❌ 失敗。ログ:')
    print(subprocess.run(['tail', '-30', '/tmp/llm.log'], capture_output=True, text=True).stdout)

---
## ⑤ llama.cpp 埋め込みサーバーの起動（port 18081）

In [ ]:
import subprocess, sys, os, time, urllib.request, glob

model_files = glob.glob('/tmp/models/*.gguf')
EMBED_PATH = next((f for f in model_files if 'embed' in f.lower()), None)
EMBED_PORT = 18081

subprocess.run(['fuser', '-k', str(EMBED_PORT) + '/tcp'], capture_output=True)
time.sleep(1)

r = subprocess.run([sys.executable, '-m', 'llama_cpp.server', '--help'],
                   capture_output=True, text=True)
help_text = r.stdout + r.stderr
use_n_gpu  = '--n_gpu_layers' in help_text or '--n-gpu-layers' in help_text
use_embed  = '--embedding' in help_text

cmd = [sys.executable, '-m', 'llama_cpp.server',
       '--model', EMBED_PATH, '--host', '0.0.0.0', '--port', str(EMBED_PORT),
       '--n_ctx', '512', '--n_threads', '2']
if use_n_gpu:  cmd += ['--n_gpu_layers', '0']
if use_embed:  cmd += ['--embedding', 'true']

embed_proc = subprocess.Popen(cmd,
    stdout=open('/tmp/embed.log', 'w'), stderr=subprocess.STDOUT)

print('埋め込みサーバー起動中... (最大 60 秒)')
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:' + str(EMBED_PORT) + '/v1/models', timeout=3)
        print('\n✅ 埋め込みサーバー起動完了！ port=' + str(EMBED_PORT))
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(2)
else:
    print('\n❌ 失敗。ログ:')
    print(subprocess.run(['tail', '-20', '/tmp/embed.log'], capture_output=True, text=True).stdout)

---
## ⑥ OpenSearch の起動（tar.gz バイナリ / Docker 不要）

> Colab は root で動作するため専用ユーザーを作成して起動します

In [ ]:
%%bash
OS_VERSION="2.17.1"
OS_DIR="/opt/opensearch-${OS_VERSION}"

if [ ! -d "${OS_DIR}" ]; then
    echo "OpenSearch ${OS_VERSION} をダウンロード中... （約 600MB）"
    wget -q --show-progress \
        "https://artifacts.opensearch.org/releases/bundle/opensearch/${OS_VERSION}/opensearch-${OS_VERSION}-linux-x64.tar.gz" \
        -O /tmp/opensearch.tar.gz
    tar -xzf /tmp/opensearch.tar.gz -C /opt/
    rm /tmp/opensearch.tar.gz
    echo "✅ 展開完了"
else
    echo "✅ 既にインストール済み"
fi

In [ ]:
%%bash
OS_VERSION="2.17.1"
OS_DIR="/opt/opensearch-${OS_VERSION}"

cat > "${OS_DIR}/config/opensearch.yml" << 'EOF'
cluster.name: rag-local
node.name: node-1
network.host: 0.0.0.0
http.port: 9200
discovery.type: single-node
plugins.security.disabled: true
bootstrap.memory_lock: false
action.auto_create_index: true
EOF

sed -i 's/-Xms[0-9]*[gGmM]/-Xms512m/g' "${OS_DIR}/config/jvm.options"
sed -i 's/-Xmx[0-9]*[gGmM]/-Xmx512m/g' "${OS_DIR}/config/jvm.options"
sysctl -w vm.max_map_count=262144 2>/dev/null || true
echo "✅ OpenSearch 設定完了" 

In [ ]:
import subprocess, os, time, urllib.request, json

OS_VERSION = '2.17.1'
OS_DIR = f'/opt/opensearch-{OS_VERSION}'

# Colab は root のため専用ユーザーを作成
subprocess.run(['useradd', '-m', '-s', '/bin/bash', 'opensearch'], capture_output=True)
subprocess.run(['chown', '-R', 'opensearch:opensearch', OS_DIR], check=True)
print('✅ opensearch ユーザー設定完了')

bundled_jdk = f'{OS_DIR}/jdk'
java_home = bundled_jdk if os.path.isdir(bundled_jdk) else '/usr'

os_proc = subprocess.Popen(
    ['sudo', '-u', 'opensearch', f'{OS_DIR}/bin/opensearch'],
    env={**os.environ, 'JAVA_HOME': java_home,
         'OPENSEARCH_JAVA_OPTS': '-Xms512m -Xmx512m',
         'HOME': '/home/opensearch'},
    stdout=open('/tmp/opensearch.log', 'w'), stderr=subprocess.STDOUT,
)

print('OpenSearch 起動中... (最大 90 秒)')
for i in range(45):
    try:
        data = json.loads(urllib.request.urlopen('http://localhost:9200', timeout=3).read())
        print('\n✅ OpenSearch 起動完了！ v' + data['version']['number'])
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(2)
else:
    print('\n❌ タイムアウト。ログ:')
    print(subprocess.run(['tail', '-20', '/tmp/opensearch.log'], capture_output=True, text=True).stdout)

---
## ⑦ RAG サーバーの起動（FastAPI, port 8100）

```
ユーザー → POST /ask  {question} → RAG サーバー → OpenSearch + llama.cpp → {answer, sources}
```

In [ ]:
%%writefile /tmp/rag_server.py
import urllib.request, json, time, glob, os
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="RAG Server", description="llama.cpp + OpenSearch による完全ローカル RAG")

app.add_middleware(CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

OS_URL       = "http://localhost:9200"
LLM_BASE     = "http://localhost:18080"
INDEX        = "rag-docs"

model_files  = glob.glob("/tmp/models/*.gguf")
LLM_MODEL_ID = next((f for f in model_files if "embed" not in f.lower()), "")

def os_req(method, path, data=None):
    body = json.dumps(data, ensure_ascii=False).encode("utf-8") if data else None
    req  = urllib.request.Request(
        OS_URL + path, data=body,
        headers={"Content-Type": "application/json; charset=utf-8"},
        method=method)
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            return json.loads(r.read())
    except urllib.error.HTTPError as e:
        return json.loads(e.read())


class AskRequest(BaseModel):
    question: str
    top_k: int = 3

class AskResponse(BaseModel):
    answer: str
    sources: list
    elapsed: float

class DocRequest(BaseModel):
    title: str
    content: str


@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/docs_list")
def docs_list():
    r = os_req("GET", f"/{INDEX}/_search",
               {"query": {"match_all": {}}, "size": 100,
                "_source": ["title"], "sort": [{"_id": "asc"}]})
    hits = r.get("hits", {}).get("hits", [])
    return {"count": len(hits),
            "docs": [{"id": h["_id"], "title": h["_source"]["title"]} for h in hits]}

@app.post("/add_doc")
def add_doc(req: DocRequest):
    r = os_req("POST", f"/{INDEX}/_doc", {"title": req.title, "content": req.content})
    os_req("POST", f"/{INDEX}/_refresh")
    return {"result": r.get("result"), "id": r.get("_id")}

@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest):
    hits = os_req("POST", f"/{INDEX}/_search", {
        "query": {"multi_match": {"query": req.question, "fields": ["title^2", "content"]}},
        "size": req.top_k
    }).get("hits", {}).get("hits", [])

    if not hits:
        return AskResponse(answer="関連ドキュメントが見つかりませんでした。", sources=[], elapsed=0.0)

    context = "\n\n".join(
        f"[{h['_source']['title']}]\n{h['_source']['content']}" for h in hits
    )

    payload = {
        "model": LLM_MODEL_ID,
        "messages": [
            {"role": "system",
             "content": "以下のドキュメントを参考に、質問に日本語で簡潔に答えてください。"},
            {"role": "user",
             "content": f"=== ドキュメント ===\n{context}\n\n=== 質問 ===\n{req.question}"}
        ],
        "max_tokens": 400,
        "temperature": 0.3,
    }
    api_req = urllib.request.Request(
        LLM_BASE + "/v1/chat/completions",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"})

    t0 = time.time()
    with urllib.request.urlopen(api_req, timeout=300) as resp:
        result = json.loads(resp.read())
    elapsed = round(time.time() - t0, 2)

    answer  = result["choices"][0]["message"]["content"]
    sources = [h["_source"]["title"] for h in hits]
    return AskResponse(answer=answer, sources=sources, elapsed=elapsed)


In [ ]:
import subprocess, sys, time, urllib.request, os

# OpenSearch に RAG 用インデックスを作成
import json

def os_req(method, path, data=None):
    body = json.dumps(data, ensure_ascii=False).encode('utf-8') if data else None
    req  = urllib.request.Request(
        'http://localhost:9200' + path, data=body,
        headers={'Content-Type': 'application/json; charset=utf-8'},
        method=method)
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            return json.loads(r.read())
    except urllib.error.HTTPError as e:
        return json.loads(e.read())

os_req('DELETE', '/rag-docs')
r = os_req('PUT', '/rag-docs', {
    'settings': {'number_of_shards': 1, 'number_of_replicas': 0},
    'mappings': {'properties': {
        'title':   {'type': 'text'},
        'content': {'type': 'text'},
    }}
})
print('インデックス作成:', r.get('acknowledged'))

# RAG サーバー起動
RAG_PORT = 8100
subprocess.run(['fuser', '-k', str(RAG_PORT) + '/tcp'], capture_output=True)
time.sleep(1)

rag_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'rag_server:app',
     '--host', '0.0.0.0', '--port', str(RAG_PORT), '--log-level', 'warning'],
    cwd='/tmp',
    stdout=open('/tmp/rag_server.log', 'w'),
    stderr=subprocess.STDOUT,
)

print('RAG サーバー起動中...')
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:' + str(RAG_PORT) + '/health', timeout=3)
        print('\n✅ RAG サーバー起動完了！ port=' + str(RAG_PORT))
        print('   API Docs: http://localhost:' + str(RAG_PORT) + '/docs')
        break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(2)
else:
    print('\n❌ 失敗。ログ:')
    print(subprocess.run(['tail', '-20', '/tmp/rag_server.log'], capture_output=True, text=True).stdout)

---
## ⑧ サンプルドキュメントの登録

In [ ]:
import urllib.request, json

RAG_BASE = 'http://localhost:8100'

def rag_req(method, path, data=None):
    body = json.dumps(data, ensure_ascii=False).encode('utf-8') if data else None
    req  = urllib.request.Request(
        RAG_BASE + path, data=body,
        headers={'Content-Type': 'application/json; charset=utf-8'},
        method=method)
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())

docs = [
    ('RAGとは',
     'RAGはRetrieval Augmented Generationの略で検索と生成を組み合わせた技術です。'
     '質問に関連するドキュメントをキーワード検索で取得しLLMのプロンプトに含めて回答を生成します。'
     'LLMの知識カットオフを超えた最新情報や独自データへの対応が可能になります。'),
    ('llama.cppとは',
     'llama.cppはC++で実装された軽量LLM推論ライブラリです。'
     'GGUFフォーマットのモデルをCPUのみで動作させることができます。'
     'pip install llama-cpp-python でインストールでき、OpenAI互換のREST APIサーバーも提供します。'),
    ('OpenSearchとは',
     'OpenSearchはAWSが開発したオープンソースの検索エンジンです。'
     'Apache Luceneをベースにしており全文検索とk-NNベクトル検索の両方に対応しています。'
     'HNSWアルゴリズムによる近似最近傍探索が可能でRAGシステムのベクトルDBとして広く使われています。'),
    ('Qwen2.5モデルについて',
     'Qwen2.5はAlibabaが開発したオープンソースのLLMシリーズです。'
     '3Bパラメータの小型モデルでもツールコール（Function Calling）に対応しています。'
     '日本語を含む多言語に対応しCPUのみでも実用的な速度で動作します。'),
]

for title, content in docs:
    r = rag_req('POST', '/add_doc', {'title': title, 'content': content})
    print(f'登録: {title} → {r.get("result")}')

r = rag_req('GET', '/docs_list')
print(f'\n✅ 登録済みドキュメント数: {r["count"]} 件')

---
## ⑨ RAG デモ（RAG サーバー経由）

```
ユーザー → POST /ask → RAG サーバー → OpenSearch + llama.cpp → 回答
```

In [ ]:
import urllib.request, json

RAG_BASE = 'http://localhost:8100'

def ask(question):
    req = urllib.request.Request(
        RAG_BASE + '/ask',
        data=json.dumps({'question': question}).encode(),
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read())

questions = [
    'RAGとはどのような技術ですか？',
    'llama.cppをCPUで動かすにはどうすればいいですか？',
    'OpenSearchのベクトル検索に使われるアルゴリズムは？',
    'Qwen2.5は日本語に対応していますか？',
]

for q in questions:
    print(f'\n{"="*62}')
    print(f'❓ 質問: {q}')
    print('   生成中...', end='', flush=True)
    resp = ask(q)
    print(f'\r💬 回答 ({resp["elapsed"]}秒):')
    print(resp['answer'])
    print(f'\n📄 参照: {" / ".join(resp["sources"])}')

---
## ⑩ 外部公開（Cloudflare Tunnel）

> アカウント不要・インストールだけで使えます。

In [ ]:
import subprocess, sys, time, re

# cloudflared をインストール
r = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not r.stdout.strip():
    print('cloudflared をインストール中...')
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
    print('✅ cloudflared インストール完了')
else:
    print('✅ cloudflared 既にインストール済み')

# RAG サーバーをトンネル公開
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8100'],
    stdout=open('/tmp/cloudflared.log', 'w'),
    stderr=subprocess.STDOUT,
)

print('トンネル開通中...')
PUBLIC_URL = None
for i in range(30):
    try:
        log = open('/tmp/cloudflared.log').read()
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', log)
        if match:
            PUBLIC_URL = match.group(0)
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(2)

if PUBLIC_URL:
    print('\n\n🌐 RAG サーバー公開 URL:', PUBLIC_URL)
    print('\n使い方:')
    print(f'  # ヘルスチェック')
    print(f'  curl {PUBLIC_URL}/health')
    print(f'  # 質問する')
    print(f'  curl -X POST {PUBLIC_URL}/ask \\')
    print(f'       -H "Content-Type: application/json" \\')
    print(f'       -d \'{{"question": "RAGとは？"}}\'')
    print(f'  # API ドキュメント（ブラウザで開く）')
    print(f'  {PUBLIC_URL}/docs')
else:
    print('\n❌ URL 取得失敗。ログ確認:')
    print(open('/tmp/cloudflared.log').read()[-500:])


---
## ⑪ ヘルスチェック

In [ ]:
import urllib.request

services = [
    ('llama.cpp LLM  ', 'http://localhost:18080', '/v1/models'),
    ('llama.cpp Embed', 'http://localhost:18081', '/v1/models'),
    ('OpenSearch     ', 'http://localhost:9200',  ''),
    ('RAG サーバー   ', 'http://localhost:8100',  '/health'),
]

print(f'{"サービス":<22} {"状態":<12} URL')
print('-' * 60)
for name, base, path in services:
    try:
        urllib.request.urlopen(base + path, timeout=5)
        print(f'{name:<22} ✅ 稼働中     {base}')
    except Exception as e:
        print(f'{name:<22} ❌ 停止中     {base}')

---
## ⑫ ログ確認

In [ ]:
import subprocess

TARGET = 'RAG サーバー'  # 変更して他のログも確認できます

logs = {
    'LLM':        '/tmp/llm.log',
    'Embed':      '/tmp/embed.log',
    'OpenSearch': '/tmp/opensearch.log',
    'RAG サーバー': '/tmp/rag_server.log',
}

print(f'=== {TARGET} ログ（末尾 30 行）===')
r = subprocess.run(['tail', '-30', logs[TARGET]], capture_output=True, text=True)
print(r.stdout or '（ログなし）')

---
## ⑬ クリーンアップ

In [ ]:
CLEANUP = False  # True にして実行すると全サービスを停止

if CLEANUP:
    # cloudflared 停止
    try:
        tunnel_proc.terminate()
        print('停止: cloudflared')
    except Exception:
        pass

    for name, proc in [
        ('RAG サーバー',    rag_proc),
        ('OpenSearch',      os_proc),
        ('llama.cpp Embed', embed_proc),
        ('llama.cpp LLM',   llm_proc),
    ]:
        try:
            proc.terminate()
            proc.wait(timeout=10)
            print('停止:', name)
        except Exception as e:
            print('エラー:', name, e)
    print('\n✅ 全サービスを停止しました')
else:
    print('CLEANUP = True にして再実行すると停止できます')